# DOAR — Google Colab Runner

This notebook **only calls** the `src/` modules. All core logic lives in
Python files in the repo, so Colab and VS Code run the exact same code.

**Order:** mount Drive → clone/update repo → install → set paths →
inspect → split → train → evaluate → analyze examples → thesis outputs → copy to Drive.

> Non-diagnostic research tool. Every parent-facing output carries a disclaimer.

## 1. Mount Google Drive (optional but recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone or update the repository

In [ ]:
import os
REPO_URL = 'https://github.com/HHemaly/DOAR.git'
REPO_DIR = '/content/DOAR'
if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull
%cd $REPO_DIR

## 3. Install dependencies
Colab already has torch + torchvision; we add the light extras.

In [ ]:
!pip install -q opencv-python Pillow matplotlib scikit-learn easyocr \
    sentence-transformers deep-translator gradio tqdm

## 4. Set the dataset path
Point this at your dataset in Drive (or a local Colab copy).

In [ ]:
import os
# Example: a Combined_Drawing folder inside your Drive
DATASET = '/content/drive/MyDrive/Masters/Datasets/Combined_Drawing'
OUTPUT  = '/content/DOAR/outputs'
os.environ['DOAR_DATASET'] = DATASET
os.environ['DOAR_OUTPUT']  = OUTPUT
assert os.path.isdir(DATASET), f'Dataset not found: {DATASET}'
print('Dataset:', DATASET)

## 5. Inspect the dataset
Writes CSVs, `dataset_statistics.json`, and distribution figures.

In [ ]:
!python main.py inspect --data "$DATASET" --out "$OUTPUT"

## 6. Build a leak-safe train/val/test split (70/15/15)

In [ ]:
!python main.py split --out "$OUTPUT"

In [ ]:
import json
meta = json.load(open(f'{OUTPUT}/splits/split_meta.json'))
print('Split totals:', meta['split_totals'])
print('Leakage OK  :', meta['leakage_ok'])

## 7. Train (GPU recommended)
Transfer model (ResNet18) by default. Use `--model baseline` for the
small CNN, or `mobilenet` / `efficientnet` for comparison.

In [ ]:
!python main.py train --out "$OUTPUT" --model transfer --epochs 25 --batch-size 32

## 8. Evaluate on the untouched test split
Confusion matrix, per-class metrics, predictions_test.csv, etc.

In [ ]:
!python main.py evaluate --out "$OUTPUT"

In [ ]:
import json
m = json.load(open(f'{OUTPUT}/evaluation/metrics.json'))
print('accuracy       :', m['accuracy'])
print('balanced acc   :', m['balanced_accuracy'])
print('macro F1       :', m['macro_f1'])
print('weighted F1    :', m['weighted_f1'])

In [ ]:
from IPython.display import Image
Image(f'{OUTPUT}/evaluation/figures/confusion_matrix.png')

## 9. Analyze example drawings (interpretation pipeline)
Objective features + rules + claims + validators + safe parent answer.

In [ ]:
!python main.py analyze-dataset --data "$DATASET" --out "$OUTPUT" --max 3

## 10. Collate thesis figures and tables

In [ ]:
!python main.py thesis --out "$OUTPUT"

## 11. Copy outputs back to Drive

In [ ]:
import shutil, os
DEST = '/content/drive/MyDrive/Masters/DOAR_outputs'
os.makedirs(DEST, exist_ok=True)
shutil.copytree(OUTPUT, DEST, dirs_exist_ok=True)
print('Copied outputs to', DEST)